# NuCLS E2 · Cierre de reserva y estadística

**Usar cuando termine la cola actual.** Selecciona GPU, pulsa **Ejecutar todo** y autoriza la misma cuenta de Drive que contiene los resultados. No necesitas copiar otros códigos.

Este cuaderno verifica **13 modelos × 5 semillas (6101–6105)**. No entrena, no modifica E1 y no usa la búsqueda 7001 como repetición del piloto. Si falta evidencia o una ejecución sigue marcada activa, se detiene antes de abrir la reserva.

Las predicciones terminadas se conservan y verifican para reanudar. La reserva es la heredada y ya fue observada en E1. El cierre genera un informe y un ZIP; no declara cumplidos los requisitos que aún dependan de entrenamientos o experimentos ausentes.


## 1. Conectar Drive y comprobar que terminó el desarrollo

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('MPLCONFIGDIR', '/content/nucls_cierre_mpl')
from google.colab import drive
MOUNT = Path('/content/drive')
if not (MOUNT / 'MyDrive').is_dir():
    if MOUNT.is_dir() and any(MOUNT.iterdir()):
        raise RuntimeError('El punto de montaje contiene archivos pero no MyDrive. No se borró nada; reconecta el entorno antes de continuar.')
    drive.mount(str(MOUNT))
DRIVE = MOUNT / 'MyDrive'
assert DRIVE.is_dir(), 'Drive no está disponible.'
for folder in ['NuCLS_E2_ADICIONALES_EXPLORATORIO', 'NuCLS_E2_TRES_NUEVOS_V1', 'NuCLS_E2_COLA_UNICA_V1', 'NuCLS_E2_CIERRE_RESERVA_V1']:
    lock = DRIVE / folder / 'EJECUCION_ACTIVA.json'
    if lock.exists():
        # Solo la marca del cierre propio y de un proceso muerto en este host podrá recuperarse dentro del programa.
        import platform
        info = json.loads(lock.read_text())
        recoverable = False
        if folder == 'NuCLS_E2_CIERRE_RESERVA_V1' and info.get('host') == platform.node():
            try: os.kill(int(info['pid']), 0)
            except ProcessLookupError: recoverable = True
        if not recoverable:
            raise RuntimeError('Existe una marca de ejecución: ' + str(lock) + '. Verifica que terminó esa sesión; no se borró ni se inició otra copia.')
print('Drive disponible. Comprobación inicial completada; aún no se evaluó reserva.')


## 2. Preparar el código de cierre y sus dependencias

In [ ]:
# No se actualiza PyTorch ni se reinstala la GPU.
import importlib.util
import base64, hashlib, io, zipfile
requirements = []
for module, package in [('scipy','scipy'),('sklearn','scikit-learn'),('matplotlib','matplotlib'),('joblib','joblib')]:
    if importlib.util.find_spec(module) is None: requirements.append(package)
try: timm_version = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError: timm_version = None
if timm_version != '1.0.22': requirements.append('timm==1.0.22')
classic = DRIVE / 'NuCLS_E2_REFERENCIAS_SIMPLES' / 'resultado.json'
if classic.is_file():
    wanted = json.loads(classic.read_text()).get('sklearn')
    if wanted:
        requirements = [r for r in requirements if r != 'scikit-learn']
        try: current = importlib.metadata.version('scikit-learn')
        except importlib.metadata.PackageNotFoundError: current = None
        if current != wanted: requirements.append('scikit-learn==' + wanted)
if requirements:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + requirements, check=True)
CODIGO = Path('/content/nucls_e2_cierre_reserva_v1_codigo')
CODIGO.mkdir(exist_ok=True)
raw = base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest() == 'd6d6e84818216e7c68d1bcbc5042dd01159a1311de2315a69df81327344efe51'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target = (CODIGO / name).resolve()
        assert target.is_relative_to(CODIGO.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(name))
print('Código íntegro verificado. No se inició entrenamiento.')


## 3. Ejecutar el cierre completo

Primero verifica las 65 corridas y reproduce validación. Luego congela las entradas, evalúa reserva, calcula la brecha con inferencia sobre train y genera la estadística. Puede tardar varias horas según GPU; no contiene entrenamiento.

Cada cierre de corrida genera un respaldo. No vuelvas a pulsar ejecutar mientras esta celda esté activa. Si se interrumpe, vuelve a ejecutar este mismo cuaderno; una marca de otra sesión exige verificarla antes de continuar.


In [ ]:
import datetime, signal
SALIDA = DRIVE / 'NuCLS_E2_CIERRE_RESERVA_V1'
assert not SALIDA.is_symlink(), 'La salida no puede ser un enlace a otra carpeta.'
SALIDA.mkdir(exist_ok=True)
log = SALIDA / ('log_cierre_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.txt')
with log.open('w') as registro:
    proceso = subprocess.Popen([sys.executable, '-u', str(CODIGO / 'cierre_unico.py'), '--drive', str(DRIVE)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for linea in proceso.stdout:
            print(linea, end='', flush=True)
            registro.write(linea); registro.flush()
        retorno = proceso.wait()
    except KeyboardInterrupt:
        proceso.send_signal(signal.SIGINT)
        try: proceso.wait(timeout=60)
        except subprocess.TimeoutExpired:
            proceso.terminate(); proceso.wait(timeout=30)
        raise
print('Log:', log)
if retorno != 0:
    raise RuntimeError('El cierre se detuvo con evidencia conservada. Lee el diagnóstico anterior; no borres resultados ni bloqueos y no cambies de cuaderno.')


## 4. Recoger el informe y las evidencias

In [ ]:
estado = SALIDA / 'estado.json'
assert estado.is_file(), 'No hay cierre registrado.'
resultado = json.loads(estado.read_text())
print(json.dumps(resultado, ensure_ascii=False, indent=2))
assert resultado.get('estado') == 'CIERRE_RESERVA_Y_ESTADISTICA_COMPLETADO', 'El cierre aún no terminó.'
# Refrescar ZIP una vez cerrado el registro de la celda anterior.
import zipfile
zip_final = DRIVE / 'NuCLS_E2_CIERRE_RESERVA_V1_RESULTADOS.zip'
temporal = zip_final.with_suffix('.tmp')
with zipfile.ZipFile(temporal, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SALIDA.rglob('*')):
        if p.is_file() and p.name != 'EJECUCION_ACTIVA.json' and p.suffix in ['.json','.csv','.npz','.html','.png','.txt','.py']:
            z.write(p, str(p.relative_to(SALIDA)))
temporal.replace(zip_final)
print('ZIP FINAL:', zip_final)
print('INFORME:', SALIDA / 'informe.html')
print('Guarda este notebook con salidas: Archivo → Descargar → Descargar .ipynb.')
print('Descarga el ZIP desde Drive. Extrae todo antes de abrir informe.html.')


## Interpretación y límites

- Bootstrap de **2.000 remuestreos por paciente**, con las mismas muestras para los modelos. Las semillas permanecen fijas en ese intervalo.
- Contrastes entre las cinco semillas condicionados a la partición; Holm para los diez modelos frente a ResNet50. Wilcoxon bilateral exacto tiene p mínimo 0,0625 con cinco pares.
- DeLong, McNemar, binomial y AUC frente a 0,5 por recorte son diagnósticos bajo independencia. No sustituyen los contrastes agrupados por paciente.
- La referencia logística se carga de lo ya entrenado. Si falta, se declara pendiente; no se ajusta al ver la reserva.
- E1 permanece intacto. Su reserva no se presenta como un test nunca observado.
- El cierre no crea resultados de pilotos ausentes, ablaciones, PCam, robustez, Grad-CAM ni cobertura conforme.
